In [ ]:
# This notebook learns a complete small audio-classification pipeline.
# It intentionally starts from raw waveforms so the encoder is easy to inspect.
# Install matching PyTorch and torchaudio versions before running this notebook.

from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Reproducibility makes learning experiments easier to compare.
torch.manual_seed(7)

# Use a GPU when PyTorch can see one; the notebook also works on CPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')
print(f'torchaudio version: {torchaudio.__version__}')

In [ ]:
# These are the only command labels we will classify in this beginner project.
# Restricting the task keeps training time and model size manageable.
LABELS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABELS)}

# Speech Commands recordings are normally 16 kHz.
# One second at this rate contains exactly 16,000 samples.
SAMPLE_RATE = 16_000
NUM_SAMPLES = SAMPLE_RATE
BATCH_SIZE = 64
NUM_EPOCHS = 2
LEARNING_RATE = 1e-3

# The dataset will be downloaded here the first time the next cell runs.
DATA_ROOT = Path('data')
DATA_ROOT.mkdir(exist_ok=True)

In [ ]:
# This dataset wrapper uses the official Speech Commands split files.
# It also filters the full dataset down to the ten labels above.
class TenCommandSpeechDataset(Dataset):
    def __init__(self, root, subset):
        # download=True fetches Speech Commands v0.02 only when it is absent.
        self.base = torchaudio.datasets.SPEECHCOMMANDS(
            root=str(root),
            download=True,
            subset=subset,
        )

        # Each item path contains its label as its parent directory name.
        # _walker is the dataset's list of audio files for this official split.
        self.indices = [
            index for index, path in enumerate(self.base._walker)
            if Path(path).parent.name in LABEL_TO_INDEX
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        # waveform has shape [channels, samples] and is usually mono.
        waveform, sample_rate, label, _, _ = self.base[self.indices[index]]

        # Average channels so every recording becomes one mono waveform.
        waveform = waveform.mean(dim=0, keepdim=True)

        # This check makes the notebook robust if a recording differs in rate.
        if sample_rate != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)

        # A fixed length lets DataLoader stack clips into a batch tensor.
        # Short clips get zero padding; long clips are cut to the first second.
        if waveform.size(1) < NUM_SAMPLES:
            waveform = torch.nn.functional.pad(waveform, (0, NUM_SAMPLES - waveform.size(1)))
        else:
            waveform = waveform[:, :NUM_SAMPLES]

        return waveform, LABEL_TO_INDEX[label]


train_dataset = TenCommandSpeechDataset(DATA_ROOT, subset='training')
validation_dataset = TenCommandSpeechDataset(DATA_ROOT, subset='validation')
test_dataset = TenCommandSpeechDataset(DATA_ROOT, subset='testing')

print(f'Training examples: {len(train_dataset):,}')
print(f'Validation examples: {len(validation_dataset):,}')
print(f'Test examples: {len(test_dataset):,}')
print(f'Classes: {LABELS}')

In [ ]:
# Visualizing one raw waveform connects the audio file to the input tensor.
example_waveform, example_label = train_dataset[0]
time = torch.arange(NUM_SAMPLES) / SAMPLE_RATE

plt.figure(figsize=(12, 3))
plt.plot(time, example_waveform.squeeze().numpy(), linewidth=0.7)
plt.title(f'One raw waveform: {LABELS[example_label]}')
plt.xlabel('Time in seconds')
plt.ylabel('Amplitude')
plt.show()

# Each batch has shape [batch, 1 channel, 16000 samples].
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

waveforms, labels = next(iter(train_loader))
print('Waveform batch shape:', tuple(waveforms.shape))
print('Label batch shape:', tuple(labels.shape))

In [ ]:
# A 1D convolution is the first-level audio encoder.
# It moves learned filters along time and converts nearby waveform samples into features.
class SimpleAudioClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # The encoder produces increasingly abstract feature maps over time.
        # Stride and max pooling shorten the time axis after each stage.
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=80, stride=4, padding=38),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=9, padding=4),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=9, padding=4),
            nn.ReLU(),
        )

        # Global average pooling makes every clip into one 64-number embedding.
        self.global_pool = nn.AdaptiveAvgPool1d(output_size=1)

        # The classifier turns the embedding into one logit per command.
        self.classifier = nn.Linear(in_features=64, out_features=num_classes)

    def forward(self, waveform):
        # Input: [batch, 1, samples]
        features = self.encoder(waveform)
        # features is [batch, 64, shorter_time_axis] at this point.
        embedding = self.global_pool(features).squeeze(-1)
        # Output: [batch, num_classes], containing unnormalized class scores.
        logits = self.classifier(embedding)
        return logits, embedding


model = SimpleAudioClassifier(num_classes=len(LABELS)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# This number is the count of trainable weights, not the number of audio samples.
parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(model)
print(f'Trainable parameters: {parameter_count:,}')

In [ ]:
# This evaluation function never updates weights.
# no_grad reduces memory use because PyTorch does not build a backward graph.
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    for waveforms, labels in loader:
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        logits, _ = model(waveforms)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return correct / total


# Training alternates between a forward pass, loss calculation, and weight update.
history = {'train_loss': [], 'validation_accuracy': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    progress = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{NUM_EPOCHS}')
    for waveforms, labels in progress:
        waveforms = waveforms.to(device)
        labels = labels.to(device)

        # Clear gradients left from the preceding batch.
        optimizer.zero_grad()
        logits, _ = model(waveforms)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        progress.set_postfix(loss=f'{loss.item():.3f}')

    average_loss = running_loss / len(train_loader)
    validation_accuracy = evaluate(model, validation_loader)
    history['train_loss'].append(average_loss)
    history['validation_accuracy'].append(validation_accuracy)
    print(f'Epoch {epoch + 1}: loss={average_loss:.4f}, validation accuracy={validation_accuracy:.2%}')

In [ ]:
# Compare the final model only once on the untouched test split.
test_accuracy = evaluate(model, test_loader)
print(f'Test accuracy: {test_accuracy:.2%}')

# Plotting learning curves helps distinguish learning from a stalled training run.
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history['train_loss'], marker='o')
axes[0].set_title('Training loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-entropy loss')
axes[1].plot(epochs, history['validation_accuracy'], marker='o')
axes[1].set_title('Validation accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
plt.show()

In [ ]:
# Inspect one prediction and the learned 64-number embedding behind it.
model.eval()
single_waveform, true_label = test_dataset[0]

with torch.no_grad():
    logits, embedding = model(single_waveform.unsqueeze(0).to(device))
    probabilities = logits.softmax(dim=1).squeeze(0).cpu()
    predicted_label = probabilities.argmax().item()

print(f'True label: {LABELS[true_label]}')
print(f'Predicted label: {LABELS[predicted_label]}')
print(f'Embedding shape: {tuple(embedding.shape)}')

# The most likely labels make the classifier's decision easier to inspect.
top_probabilities, top_indices = probabilities.topk(3)
for probability, index in zip(top_probabilities, top_indices):
    print(f'{LABELS[index.item()]}: {probability.item():.2%}')